# ❄️🐉 cryoDRGN on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ts387/cryodrgn/blob/claude/cryodrgn-colab-notebook-5mf3p3/cryoDRGN_colab.ipynb)

**cryoDRGN** is a neural-network method for **heterogeneous cryo-EM reconstruction** — it learns a *continuous* distribution of 3D structures directly from a single-particle dataset.

This notebook walks you end-to-end on a free/Pro Colab GPU:

| Step | What happens |
|------|--------------|
| 1. Setup | Check the GPU, install cryoDRGN, mount Google&nbsp;Drive |
| 2. Inputs | Point to your particles, poses and CTF (sourced from Drive) |
| 3. Preprocess | `downsample` → `parse_pose_*` → `parse_ctf_*` |
| 4. Sanity check | `backproject_voxel` a subset and view slices |
| 5. Train | `train_vae` a heterogeneous model |
| 6. Analyze | `analyze` the latent space + view plots and volumes inline |
| 7. Filter *(optional)* | Remove junk particles and retrain — in Colab, or export for local `cryodrgn filter` |
| 8. Save | Sync results back to your Google Drive |

> **You will need**, from an upstream consensus refinement (RELION or cryoSPARC):
> - a **particle stack** — `.mrcs` / `.star` / `.cs` / `.txt`
> - a **`.star`** (RELION) **or `.cs`** (cryoSPARC) file to extract **poses** and **CTF** from
>
> Don't have data yet? Try the tutorial dataset (EMPIAR-10076) from the
> [cryoDRGN user guide](https://ez-lab.gitbook.io/cryodrgn/).

📖 Docs: <https://ez-lab.gitbook.io/cryodrgn/> · 💻 GitHub: <https://github.com/ml-struct-bio/cryodrgn> · 📄 [Zhong et al., *Nature Methods* 2021](https://doi.org/10.1038/s41592-020-01049-4)

---
### ⚙️ Before you start — turn on the GPU
**Runtime → Change runtime type → Hardware accelerator → GPU** (a T4 is fine for `D=128`; use an A100/L4 on Colab Pro for `D=256`).

Then run the cells **in order** (▶ on each, or *Runtime → Run all*). Each cell is a collapsible **form** — edit the fields on the right, no coding required.

## 1 · Setup

In [ ]:
#@title 1.1 · Check the GPU runtime { display-mode: "form" }
#@markdown Confirms a CUDA GPU is attached. If this prints **"No GPU found"**, go to
#@markdown **Runtime → Change runtime type → GPU** and re-run this cell.
import subprocess, sys

print("=" * 60)
gpu = subprocess.run(["nvidia-smi",
                      "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode == 0 and gpu.stdout.strip():
    name, mem, driver = [x.strip() for x in gpu.stdout.strip().split(",")]
    print(f"✅ GPU detected : {name}")
    print(f"   Memory       : {mem}")
    print(f"   Driver       : {driver}")
else:
    print("❌ No GPU found!")
    print("   Runtime → Change runtime type → Hardware accelerator → GPU,")
    print("   then re-run this cell. cryoDRGN training needs a GPU.")
print("=" * 60)

In [ ]:
#@title 1.2 · Install cryoDRGN { display-mode: "form" }
#@markdown Installs cryoDRGN from PyPI. Colab's pre-installed PyTorch/CUDA are kept.
#@markdown <br>• **stable** – the recommended release &nbsp;•&nbsp; **beta** – newest dev build from TestPyPI
release_channel = "stable"  #@param ["stable", "beta"]
#@markdown Optionally pin an exact version (e.g. `4.3.0`); leave blank for the latest.
version = ""  #@param {type:"string"}
#@markdown A few dependencies are pinned to versions other than Colab's defaults, so the
#@markdown runtime **restarts automatically** at the end. That is expected — just carry
#@markdown on with the next cell afterwards.
restart_after_install = True  #@param {type:"boolean"}

import subprocess, sys

pkg = "cryodrgn"
if version.strip():
    pkg = f"cryodrgn=={version.strip()}"

if release_channel == "beta":
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "-i", "https://test.pypi.org/simple/",
           "--extra-index-url", "https://pypi.org/simple/",
           "cryodrgn", "--pre"]
    if version.strip():
        cmd[cmd.index("cryodrgn")] = pkg
else:
    cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]

print("Installing", pkg, f"({release_channel} channel) — this takes ~1-2 min...\n")
ret = subprocess.run(cmd)
if ret.returncode != 0:
    raise SystemExit("❌ pip install failed — see the log above.")

print("\n✅ cryoDRGN installed.")
if restart_after_install:
    print("🔄 Restarting the runtime to finalize the install (this is normal)...")
    print("   When it reconnects, continue from cell 1.3 — do NOT re-run this cell.")
    get_ipython().kernel.do_shutdown(True)

In [ ]:
#@title 1.3 · Verify the installation { display-mode: "form" }
#@markdown Run this **after** the runtime has restarted.
import torch, cryodrgn

print(f"cryoDRGN version : {cryodrgn.__version__}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device      : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  CUDA not available — check that the GPU runtime is selected (cell 1.1).")

# quick smoke-test of the command-line entry point
import subprocess
print("\n$ cryodrgn --version")
print(subprocess.run(["cryodrgn", "--version"], capture_output=True, text=True).stdout.strip())

## 2 · Connect Google Drive

We use **two locations**, which is standard practice for cryo-EM on Colab:

- 📁 **Drive project folder** — *durable* storage for your inputs and final results. Survives disconnects.
- ⚡ **Local scratch** (`/content/...`) — *fast* disk for the downsampled stack that training reads. Wiped when the runtime ends, but mirrored to Drive so it can be restored.

We read inputs from Drive and downsample onto fast local scratch **while mirroring a durable copy (plus `pose.pkl`/`ctf.pkl`) back to Drive**; training then writes its model **directly to Drive** (so per-epoch checkpoints survive a disconnect). The net effect: nothing expensive has to be recomputed after an interruption.

In [ ]:
#@title 2.1 · Mount Google Drive { display-mode: "form" }
#@markdown Click the link that appears, pick your Google account, and paste the code
#@markdown (or approve the pop-up). Your Drive appears under `/content/drive/MyDrive`.
from google.colab import drive
drive.mount("/content/drive")
print("\n✅ Drive mounted at /content/drive/MyDrive")

In [ ]:
#@title 2.2 · Choose your project folder { display-mode: "form" }
#@markdown **`drive_project_dir`** — a folder in *your* Drive for this project (created if missing).
#@markdown Put your input files here, and final results are saved back here.
drive_project_dir = "/content/drive/MyDrive/cryodrgn_project"  #@param {type:"string"}
#@markdown **`local_work_dir`** — fast local scratch where training runs.
local_work_dir = "/content/cryodrgn_work"  #@param {type:"string"}

import os

DRIVE_DIR = os.path.abspath(drive_project_dir)
WORK_DIR = os.path.abspath(local_work_dir)
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

# Persist these across cells for the rest of the notebook.
os.environ["CRYODRGN_DRIVE_DIR"] = DRIVE_DIR
os.environ["CRYODRGN_WORK_DIR"] = WORK_DIR
os.chdir(WORK_DIR)

print(f"📁 Drive project (durable) : {DRIVE_DIR}")
print(f"⚡ Local scratch (fast)    : {WORK_DIR}")
print(f"📂 Working directory       : {os.getcwd()}")
print("\nContents of your Drive project folder:")
for f in sorted(os.listdir(DRIVE_DIR)) or ["(empty — upload your inputs here)"]:
    print("   ", f)

## 3 · Point to your input files

cryoDRGN needs three things, all derived from an upstream **consensus refinement**:

1. **Particle images** — the stack you refined (`.mrcs`, `.star`, `.cs`, or a `.txt` of `.mrcs` paths).
2. **Poses** — orientation + shift per particle, extracted from the refinement's `.star`/`.cs`.
3. **CTF parameters** — extracted from the same `.star`/`.cs`.

Fill in the paths below (they usually live inside your Drive project folder from Step 2).

In [ ]:
#@title 3.1 · Locate inputs on Drive { display-mode: "form" }
import os
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]

#@markdown **Particle stack** — path to your images (`.mrcs`/`.star`/`.cs`/`.txt`). For a cryoSPARC
#@markdown `.cs` or RELION `.star`, point this at that **same file** (reuse it as *metadata_file*).
particles = "/content/drive/MyDrive/cryodrgn_project/particles.cs"  #@param {type:"string"}

#@markdown **Metadata file** — the RELION `.star` **or** cryoSPARC `.cs` that holds poses & CTF
#@markdown (for a `.cs`/`.star` dataset, the same file as *particles* above).
metadata_file = "/content/drive/MyDrive/cryodrgn_project/particles.cs"  #@param {type:"string"}

#@markdown **`datadir`** — the folder the metadata file's internal image paths resolve against.
#@markdown Often needed for cryoSPARC: set it to the export/job folder the `.cs` blob paths are
#@markdown relative to (e.g. `.../J486_particles_0`, so `J486/reconstructed/<uid>_particles.mrc`
#@markdown resolves). Leave blank if the images already resolve next to the metadata file.
datadir = ""  #@param {type:"string"}

# make these available to later cells
os.environ["CRYODRGN_PARTICLES"] = particles
os.environ["CRYODRGN_META"] = metadata_file
os.environ["CRYODRGN_DATADIR"] = datadir

print("Checking inputs...\n")
for label, path in [("Particles", particles), ("Metadata (.star/.cs)", metadata_file)]:
    ok = os.path.exists(path)
    print(f"  {'✅' if ok else '❌'} {label}: {path}")
    if not ok:
        print("       ^ not found — fix the path above (check spelling / that it's in your Drive).")
if datadir:
    print(f"  {'✅' if os.path.isdir(datadir) else '❌'} datadir: {datadir}")

## 4 · Preprocess

Three quick commands turn your raw inputs into what `train_vae` expects:
`downsample` the images, then extract `pose.pkl` and `ctf.pkl`. All three are **saved to your
Drive project folder** and **skipped on re-run if already present**, so an interrupted session
resumes without re-generating them (the downsampled stack restores from Drive rather than being
recomputed).

In [ ]:
#@title 4.1 · Downsample the images (saved to Drive) { display-mode: "form" }
#@markdown Writes the smaller stack to **fast local disk** for training, and mirrors a **durable
#@markdown copy to Drive** so you never re-downsample — a later session just restores it from
#@markdown Drive. Smaller boxes train **much** faster — start at **128** to sanity-check and
#@markdown filter, then optionally redo at **256** (max recommended). For a cryoSPARC `.cs` the
#@markdown box/pixel size are read from the file, and `--datadir` (from 3.1) locates the images.
box_size = 128  #@param [64, 128, 256] {type:"raw"}
#@markdown Split output into chunks of this many images if you hit memory limits (`0` = off).
chunk = 0  #@param {type:"integer"}
#@markdown Keep a durable copy of the downsampled stack on Drive (recommended).
backup_to_drive = True  #@param {type:"boolean"}

import os, glob, shutil, time
particles = os.environ["CRYODRGN_PARTICLES"]
datadir = os.environ.get("CRYODRGN_DATADIR", "")
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]

stem = f"particles.{box_size}"
local_mrcs = os.path.join(WORK_DIR, stem + ".mrcs")
chunked = bool(chunk and int(chunk) > 0)
# with --chunk, downstream steps read the particles.<D>.txt index, not the .mrcs
local_stack = (os.path.splitext(local_mrcs)[0] + ".txt") if chunked else local_mrcs
drive_stack = os.path.join(DRIVE_DIR, os.path.basename(local_stack))

def _mirror(src_dir, dst_dir):
    """Copy the stack (plus any --chunk tiles) between dirs, skipping identical files."""
    os.makedirs(dst_dir, exist_ok=True)
    copied = 0
    for f in glob.glob(os.path.join(src_dir, stem + "*")):
        dst = os.path.join(dst_dir, os.path.basename(f))
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(f):
            shutil.copy2(f, dst)
            copied += 1
    return copied

if os.path.exists(local_stack):
    print(f"✔ Local stack already present — skipping downsample: {local_stack}")
elif os.path.exists(drive_stack):
    print("↧ Found the stack on Drive — restoring to local disk (no re-downsample needed)...")
    t0 = time.time(); _mirror(DRIVE_DIR, WORK_DIR)
    print(f"  restored in {time.time() - t0:.0f}s")
else:
    cmd = f'cryodrgn downsample "{particles}" -D {box_size} -o "{local_mrcs}"'
    if chunked:
        cmd += f" --chunk {int(chunk)}"
    if datadir:
        cmd += f' --datadir "{datadir}"'
    print("$", cmd, "\n")
    get_ipython().system(cmd)

if backup_to_drive:
    sz = sum(os.path.getsize(f) for f in glob.glob(os.path.join(WORK_DIR, stem + "*")))
    print(f"⇪ Mirroring stack to Drive ({sz / 1e9:.1f} GB, one-time — lets a future session "
          f"resume without re-downsampling)...")
    t0 = time.time(); n = _mirror(WORK_DIR, DRIVE_DIR)
    print(f"  {('copied ' + str(n) + ' file(s)') if n else 'already up to date'} in {time.time() - t0:.0f}s")

os.environ["CRYODRGN_DOWNSAMPLED"] = local_stack
print(f"\n✅ Training stack (local, fast): {local_stack}")
if backup_to_drive:
    print(f"✅ Durable copy on Drive:        {drive_stack}")

In [ ]:
#@title 4.2 · Parse poses → pose.pkl (saved to Drive) { display-mode: "form" }
#@markdown Written straight to your Drive project folder (small file) and skipped on re-run if it
#@markdown already exists — so you never re-parse after a disconnect.
pose_source = "RELION .star"  #@param ["RELION .star", "cryoSPARC .cs"]
#@markdown **`box_size_D`** — box size of the **consensus refinement** (the *original*, un-downsampled
#@markdown images). A recent cryoSPARC `.cs` carries this so `0` (auto) usually works; `.star` is
#@markdown auto-detected if present. Set it only if parsing fails.
box_size_D = 0  #@param {type:"integer"}

import os
meta = os.environ["CRYODRGN_META"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
pose_pkl = os.path.join(DRIVE_DIR, "pose.pkl")
os.environ["CRYODRGN_POSE"] = pose_pkl

if os.path.exists(pose_pkl):
    print(f"✔ Poses already on Drive — skipping parse: {pose_pkl}")
else:
    if pose_source == "RELION .star":
        cmd = f'cryodrgn parse_pose_star "{meta}" -o "{pose_pkl}"'
        if int(box_size_D) > 0:
            cmd += f" -D {int(box_size_D)}"
    else:
        cmd = f'cryodrgn parse_pose_csparc "{meta}" -o "{pose_pkl}"'
        if int(box_size_D) > 0:
            cmd += f" -D {int(box_size_D)}"
    print("$", cmd, "\n")
    get_ipython().system(cmd)
print(f"\n✅ Poses → {pose_pkl}")

In [ ]:
#@title 4.3 · Parse CTF → ctf.pkl (saved to Drive) { display-mode: "form" }
#@markdown Written straight to Drive and skipped on re-run if it already exists.
ctf_source = "RELION .star"  #@param ["RELION .star", "cryoSPARC .cs"]
#@markdown For `.star`/`.cs` the box size and Å/px are read from the file; set these only if they
#@markdown are missing (`0` = auto).
box_size_D = 0  #@param {type:"integer"}
apix = 0  #@param {type:"number"}

import os
meta = os.environ["CRYODRGN_META"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
ctf_pkl = os.path.join(DRIVE_DIR, "ctf.pkl")
os.environ["CRYODRGN_CTF"] = ctf_pkl

if os.path.exists(ctf_pkl):
    print(f"✔ CTF already on Drive — skipping parse: {ctf_pkl}")
else:
    if ctf_source == "RELION .star":
        cmd = f'cryodrgn parse_ctf_star "{meta}" -o "{ctf_pkl}"'
        if int(box_size_D) > 0:
            cmd += f" -D {int(box_size_D)}"
        if float(apix) > 0:
            cmd += f" --Apix {apix}"
    else:
        cmd = f'cryodrgn parse_ctf_csparc "{meta}" -o "{ctf_pkl}"'
        if int(box_size_D) > 0:
            cmd += f" -D {int(box_size_D)}"
        if float(apix) > 0:
            cmd += f" --Apix {apix}"
    print("$", cmd, "\n")
    get_ipython().system(cmd)
print(f"\n✅ CTF → {ctf_pkl}")

## 5 · (Optional) Sanity-check poses & CTF

Before spending GPU time on training, back-project a subset of particles into a 3D map.
It should look like a **low-resolution version of your consensus structure**. If it's noise,
the poses/CTF are probably mis-parsed (a common fix is toggling `uninvert_data`).

In [ ]:
#@title 5.1 · Voxel back-projection of a subset { display-mode: "form" }
#@markdown Number of particles to use (fewer = faster, noisier).
n_particles = 10000  #@param {type:"integer"}
#@markdown Tick if your particles are dark-on-light (flips the data sign).
uninvert_data = False  #@param {type:"boolean"}

import os
ds = os.environ["CRYODRGN_DOWNSAMPLED"]
pose = os.environ["CRYODRGN_POSE"]
ctf = os.environ["CRYODRGN_CTF"]
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]
bp_dir = os.path.join(WORK_DIR, "backproject")
os.environ["CRYODRGN_BACKPROJECT"] = bp_dir

cmd = (f'cryodrgn backproject_voxel "{ds}" --poses "{pose}" --ctf "{ctf}" '
       f'-o "{bp_dir}" --first {int(n_particles)}')
if uninvert_data:
    cmd += " --uninvert-data"

print("$", cmd, "\n")
get_ipython().system(cmd)
print(f"\n✅ Map → {bp_dir}/backproject.mrc")

In [ ]:
#@title 5.2 · View central slices of the back-projected map { display-mode: "form" }
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from cryodrgn.mrcfile import parse_mrc

bp_dir = os.environ["CRYODRGN_BACKPROJECT"]
hits = glob.glob(os.path.join(bp_dir, "*.mrc"))
if not hits:
    raise FileNotFoundError(f"No .mrc found in {bp_dir} — run cell 5.1 first.")

vol, _ = parse_mrc(hits[0])
D = vol.shape[0]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (axis, title) in zip(axes, [(0, "Z"), (1, "Y"), (2, "X")]):
    sl = vol.take(D // 2, axis=axis)
    ax.imshow(sl, cmap="Greys_r")
    ax.set_title(f"central {title} slice")
    ax.axis("off")
fig.suptitle(os.path.basename(hits[0]))
plt.tight_layout()
plt.show()
print("Looks like your structure? ✅ Proceed to training.\n"
      "Just noise? ❌ Re-check poses/CTF and try toggling `uninvert_data` in 5.1.")

## 6 · Train the cryoDRGN model

Now train the VAE for heterogeneous reconstruction. Model outputs (per-epoch `weights.*.pkl`,
`z.*.pkl`, `config.yaml`) are written **directly to your Drive project folder**, so they survive a
disconnect. **6.1** starts a fresh run (and refuses to overwrite an existing one); **6.2** resumes
or extends a run from its latest checkpoint — even in a brand-new session, once Steps 2–4 have been
re-run (4.1 restores the stack from Drive, 4.2/4.3 skip).

**Tips:** `zdim` 8 is a good default (use 1 for a single motion axis, ≥10 for complex mixtures).
For a first pass, 25 epochs at `D=128` is typical. On a T4, `D=128` runs roughly a few minutes/epoch
for ~100k particles; `D=256` is far slower (use `--lazy` + Colab Pro).

In [ ]:
#@title 6.1 · Configure & launch train_vae { display-mode: "form" }
#@markdown **Latent dimension** — size of the conformational latent space.
zdim = 8  #@param [1, 2, 4, 8, 10] {type:"raw"}
#@markdown **Epochs** — full passes over the dataset.
num_epochs = 25  #@param {type:"integer"}
#@markdown **Batch size** — increase to better use a big GPU (affects dynamics).
batch_size = 8  #@param [8, 16, 32] {type:"raw"}
#@markdown **Output folder name** (created inside your Drive project folder).
output_name = "00_cryodrgn128"  #@param {type:"string"}
#@markdown Dark-on-light particles? (must match what worked in Step 5)
uninvert_data = False  #@param {type:"boolean"}
#@markdown Lazy loading — stream images from disk if the stack is too big for RAM (needed at D=256).
lazy = False  #@param {type:"boolean"}
#@markdown Tick only to **start over** in a folder that already has a run (otherwise the cell stops
#@markdown and sends you to 6.2 to continue it — this guards against overwriting on a "Run all").
overwrite = False  #@param {type:"boolean"}

import os, glob
ds = os.environ["CRYODRGN_DOWNSAMPLED"]
pose = os.environ["CRYODRGN_POSE"]
ctf = os.environ["CRYODRGN_CTF"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
outdir = os.path.join(DRIVE_DIR, output_name)
os.environ["CRYODRGN_OUTDIR"] = outdir

if glob.glob(os.path.join(outdir, "weights.*.pkl")) and not overwrite:
    raise FileExistsError(
        f"{outdir} already contains checkpoints. To CONTINUE this run, use cell 6.2 (resume). "
        f"To START OVER, tick `overwrite` above or change `output_name`.")

cmd = (f'cryodrgn train_vae "{ds}" --poses "{pose}" --ctf "{ctf}" '
       f'--zdim {zdim} -n {int(num_epochs)} -b {batch_size} -o "{outdir}"')
if uninvert_data:
    cmd += " --uninvert-data"
if lazy:
    cmd += " --lazy"

print("$", cmd, "\n" + "=" * 70)
get_ipython().system(cmd)
print("=" * 70 + f"\n✅ Training complete. Model saved to: {outdir}")

In [ ]:
#@title 6.2 · Resume / extend a run — any session { display-mode: "form" }
#@markdown Continue an existing model, **including after a disconnect** — you do NOT need to have
#@markdown run 6.1 this session. Just re-run Steps 2–4 first (4.1 restores the stack from Drive,
#@markdown 4.2/4.3 skip), then point this at the run's folder. It picks up from the latest saved
#@markdown checkpoint automatically (`--load latest`) and trains up to `num_epochs` total.
output_name = "00_cryodrgn128"  #@param {type:"string"}
#@markdown Total epochs to reach (must exceed the last completed epoch).
num_epochs = 50  #@param {type:"integer"}

import os, re, glob
ds = os.environ["CRYODRGN_DOWNSAMPLED"]
pose = os.environ["CRYODRGN_POSE"]
ctf = os.environ["CRYODRGN_CTF"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
outdir = os.path.join(DRIVE_DIR, output_name)

if not os.path.exists(os.path.join(outdir, "config.yaml")):
    raise FileNotFoundError(f"No run found at {outdir} — check output_name (it must match 6.1).")

done = [int(m.group(1)) for p in glob.glob(os.path.join(outdir, "weights.*.pkl"))
        for m in [re.search(r"weights\.(\d+)\.pkl$", os.path.basename(p))] if m]
if not done:
    raise FileNotFoundError(f"No numbered checkpoints in {outdir} — nothing to resume from.")
last = max(done)
if int(num_epochs) <= last:
    raise ValueError(f"num_epochs ({num_epochs}) must exceed the last checkpoint (epoch {last}).")

import yaml
zdim = yaml.safe_load(open(os.path.join(outdir, "config.yaml")))["model_args"]["zdim"]
os.environ["CRYODRGN_OUTDIR"] = outdir  # so Steps 7-9 target this run

cmd = (f'cryodrgn train_vae "{ds}" --poses "{pose}" --ctf "{ctf}" '
       f'--zdim {zdim} -n {int(num_epochs)} -o "{outdir}" --load latest')
print(f"Resuming '{output_name}' from epoch {last} → {int(num_epochs)}")
print("$", cmd, "\n" + "=" * 70)
get_ipython().system(cmd)
print("=" * 70 + "\n✅ Done. Re-run Step 7 (7.1) to analyze the extended model.")

## 7 · Analyze the results

`cryodrgn analyze` visualizes the latent space (PCA + UMAP), then generates representative
volumes by k-means-sampling the latent space and traversing its principal components.

In [ ]:
#@title 7.1 · Run cryodrgn analyze { display-mode: "form" }
#@markdown Epoch to analyze — leave at **-1** to auto-pick the latest saved epoch.
epoch = -1  #@param {type:"integer"}
#@markdown Number of k-means volumes to generate.
ksample = 20  #@param {type:"integer"}
#@markdown Pixel size (Å/px) written into volume headers (`0` = read from ctf.pkl / default 1).
apix = 0  #@param {type:"number"}

import os, re, glob
outdir = os.environ["CRYODRGN_OUTDIR"]

if int(epoch) < 0:  # auto-detect latest z.N.pkl
    epochs = []
    for p in glob.glob(os.path.join(outdir, "z.*.pkl")):
        m = re.search(r"z\.(\d+)\.pkl$", os.path.basename(p))
        if m:
            epochs.append(int(m.group(1)))
    if not epochs:
        raise FileNotFoundError(f"No z.N.pkl checkpoints in {outdir} — has 6.1 finished?")
    epoch = max(epochs)
    print(f"Auto-selected latest epoch: {epoch}")

os.environ["CRYODRGN_EPOCH"] = str(int(epoch))
cmd = f'cryodrgn analyze "{outdir}" {int(epoch)} --ksample {int(ksample)}'
if float(apix) > 0:
    cmd += f" --Apix {apix}"

print("$", cmd, "\n" + "=" * 70)
get_ipython().system(cmd)
print("=" * 70 + f"\n✅ Analysis → {outdir}/analyze.{int(epoch)}")

In [ ]:
#@title 7.2 · View latent-space plots inline { display-mode: "form" }
import os, glob
from IPython.display import Image, display, Markdown

outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ["CRYODRGN_EPOCH"]
adir = os.path.join(outdir, f"analyze.{epoch}")

for fname, caption in [
    ("z_pca.png", "**PCA** of the latent embeddings (colored by k-means cluster)"),
    ("umap.png", "**UMAP** of the latent embeddings"),
    ("z_pca_marginals.png", "PCA with marginal distributions"),
    ("umap_marginals.png", "UMAP with marginal distributions"),
    (f"learning_curve_epoch{epoch}.png", "Training loss curve"),
]:
    path = os.path.join(adir, fname)
    if os.path.exists(path):
        display(Markdown(caption))
        display(Image(path, width=520))
    else:
        # k-means subfolder holds some variants
        alt = glob.glob(os.path.join(adir, "kmeans*", fname))
        if alt:
            display(Markdown(caption))
            display(Image(alt[0], width=520))

In [ ]:
#@title 7.3 · Interactive 3D view of a generated volume { display-mode: "form" }
#@markdown Renders one of the k-means representative maps as a 3D isosurface you can rotate.
#@markdown Change `volume_index` (0 … ksample-1) to inspect different structures.
volume_index = 0  #@param {type:"integer"}
#@markdown Isosurface threshold as a percentile of density (higher = tighter surface).
iso_percentile = 99.0  #@param {type:"slider", min:90, max:99.9, step:0.1}
#@markdown Downsample the box for a snappier render.
display_box = 64  #@param [48, 64, 96] {type:"raw"}

import os, glob
import numpy as np
import plotly.graph_objects as go
from cryodrgn.mrcfile import parse_mrc

outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ["CRYODRGN_EPOCH"]
adir = os.path.join(outdir, f"analyze.{epoch}")

vols = sorted(glob.glob(os.path.join(adir, "kmeans*", "vol_*.mrc")))
if not vols:
    raise FileNotFoundError(f"No k-means volumes in {adir} — run 7.1 without --skip-vol.")
vol_path = vols[int(volume_index) % len(vols)]
vol, _ = parse_mrc(vol_path)

# light box downsampling for display
D = vol.shape[0]
if D > int(display_box):
    step = int(round(D / int(display_box)))
    vol = vol[::step, ::step, ::step]
D = vol.shape[0]

x, y, z = np.mgrid[0:D, 0:D, 0:D]
iso = float(np.percentile(vol, iso_percentile))
fig = go.Figure(go.Isosurface(
    x=x.flatten(), y=y.flatten(), z=z.flatten(), value=vol.flatten(),
    isomin=iso, isomax=float(vol.max()),
    surface_count=1, colorscale="Greys", showscale=False, caps=dict(x_show=False, y_show=False, z_show=False),
))
fig.update_layout(title=os.path.basename(vol_path), width=560, height=560,
                  scene=dict(xaxis_visible=False, yaxis_visible=False, zaxis_visible=False))
fig.show()
print(f"Showing {vol_path}  ({len(vols)} volumes available; set volume_index 0..{len(vols)-1})")

## 8 · (Optional) Filter particles & retrain

`analyze` usually reveals junk particles (bad clusters, latent-space outliers) that are worth
removing before retraining on the clean subset. cryoDRGN's *interactive* lasso filter
(`cryodrgn filter` and the filtering notebook) forces matplotlib's `TkAgg` desktop backend,
which Colab can't provide — so there are **two routes**, and every selection produces an
`indices.pkl` that feeds the same `--ind` retrain (8.3):

- **(a) Select in Colab → retrain.** Either **deterministic rules** (8.1 — cluster IDs, ‖z‖
  outliers, or a PC/UMAP range; reproducible and always works) or an **experimental interactive
  lasso** (8.2 — draw right on the plot). Then retrain on the result with 8.3.
- **(b) Interactive filtering on your local machine (8.4).** Package everything `cryodrgn filter`
  (or the local filtering notebook) needs into one zip on Drive, lasso locally, then bring
  `indices.pkl` back and retrain with 8.3.

> These cells assume the **first** filtering pass (model trained on the full stack). Iterative
> multi-round filtering needs index composition — see the
> [user guide](https://ez-lab.gitbook.io/cryodrgn/).

In [ ]:
#@title 8.1 · Deterministic selection → indices.pkl (in Colab) { display-mode: "form" }
#@markdown Pick particles by a **rule** and write `indices.pkl` into your model folder.
method = "Keep k-means clusters"  #@param ["Keep k-means clusters", "Remove z-norm outliers", "Keep by axis range"]
#@markdown • *Keep k-means clusters* — comma-separated cluster IDs to keep (see the k-means plot in 7.2).
clusters_to_keep = "0,1,2"  #@param {type:"string"}
#@markdown • *Remove z-norm outliers* — drop particles whose ‖z‖ exceeds mean + (this)·std.
outlier_std = 2.0  #@param {type:"number"}
#@markdown • *Keep by axis range* — keep particles whose coordinate lies within [min, max].
axis = "PC1"  #@param ["PC1", "PC2", "UMAP1", "UMAP2"]
axis_min = -3.0  #@param {type:"number"}
axis_max = 3.0  #@param {type:"number"}

import os, glob
import numpy as np
import matplotlib.pyplot as plt
from cryodrgn import analysis, utils

outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ["CRYODRGN_EPOCH"]
z = utils.load_pkl(os.path.join(outdir, f"z.{epoch}.pkl"))
z = z.reshape(z.shape[0], -1)
N = z.shape[0]
adir = os.path.join(outdir, f"analyze.{epoch}")

# PCA is always available; UMAP only exists for zdim > 2
pc = analysis.run_pca(z)[0] if z.shape[1] > 1 else z
umap_fl = os.path.join(adir, "umap.pkl")
umap = utils.load_pkl(umap_fl) if os.path.exists(umap_fl) else None

if method == "Keep k-means clusters":
    lbls = glob.glob(os.path.join(adir, "kmeans*", "labels.pkl"))
    if not lbls:
        raise FileNotFoundError("No k-means labels found — run 7.1 (analyze) first.")
    labels = utils.load_pkl(lbls[0])
    keep = [int(x) for x in clusters_to_keep.split(",") if x.strip() != ""]
    mask = np.isin(labels, keep)
elif method == "Remove z-norm outliers":
    znorm = np.linalg.norm(z, axis=1)
    mask = znorm <= (znorm.mean() + float(outlier_std) * znorm.std())
else:
    arr, col = {"PC1": (pc, 0), "PC2": (pc, 1), "UMAP1": (umap, 0), "UMAP2": (umap, 1)}[axis]
    if arr is None or col >= arr.shape[1]:
        raise ValueError(f"{axis} is unavailable for this model (UMAP needs zdim>2; PC2 needs zdim>=2).")
    coord = arr[:, col]
    mask = (coord >= float(axis_min)) & (coord <= float(axis_max))

ind_keep = np.where(mask)[0]
out_ind = os.path.join(outdir, "indices.pkl")
utils.save_pkl(ind_keep, out_ind)
print(f"Keeping {len(ind_keep)} / {N} particles ({100 * len(ind_keep) / N:.1f}%).")
print(f"✅ Saved → {out_ind}   (retrain with cell 8.3)")

# Show the split wherever a 2-D embedding is available
panels = [(X, name) for X, name in [(pc, "PCA"), (umap, "UMAP")]
          if X is not None and X.shape[1] >= 2]
if panels:
    fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 4.5), squeeze=False)
    for ax, (X, name) in zip(axes[0], panels):
        ax.scatter(X[~mask, 0], X[~mask, 1], s=2, alpha=.2, color="lightgray", rasterized=True, label="removed")
        ax.scatter(X[mask, 0], X[mask, 1], s=2, alpha=.3, color="tab:blue", rasterized=True, label="kept")
        ax.set_title(name); ax.set_xticks([]); ax.set_yticks([]); ax.legend(markerscale=4, loc="best")
    plt.tight_layout(); plt.show()

In [ ]:
#@title 8.2 · Interactive lasso selection (in Colab, experimental) → indices.pkl { display-mode: "form" }
#@markdown Draw a lasso directly on the latent scatter; on release the selection saves to
#@markdown `indices.pkl` automatically (watch for the printed confirmation). This uses Colab's
#@markdown JS→Python bridge instead of plotly's `FigureWidget` callback, which does **not** fire
#@markdown in Colab. **If nothing prints when you lasso, your session's bridge isn't wired —
#@markdown just use the deterministic cell 8.1 instead.**
max_display_points = 50000  #@param {type:"integer"}

import os, glob
import numpy as np
import plotly.graph_objects as go
from google.colab import output
from IPython.display import HTML, display
from cryodrgn import analysis, utils

outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ["CRYODRGN_EPOCH"]
adir = os.path.join(outdir, f"analyze.{epoch}")
z = utils.load_pkl(os.path.join(outdir, f"z.{epoch}.pkl"))
z = z.reshape(z.shape[0], -1)
N = z.shape[0]

# 2-D embedding for the scatter: prefer UMAP (zdim > 2), else fall back to PCA
umap_fl = os.path.join(adir, "umap.pkl")
if os.path.exists(umap_fl):
    emb, emb_name = utils.load_pkl(umap_fl), "UMAP"
elif z.shape[1] >= 2:
    emb, emb_name = analysis.run_pca(z)[0][:, :2], "PCA"
else:
    raise ValueError("Need zdim>=2 (or a completed UMAP) for a 2-D lasso — use cell 8.1 instead.")

lbls = glob.glob(os.path.join(adir, "kmeans*", "labels.pkl"))
labels = utils.load_pkl(lbls[0]) if lbls else np.zeros(N, dtype=int)

# subsample for a responsive browser plot; keep the map from display index -> original index
rng = np.random.default_rng(0)
sub = (np.sort(rng.choice(N, int(max_display_points), replace=False))
       if N > int(max_display_points) else np.arange(N))

out_ind = os.path.join(outdir, "indices.pkl")
def _save_selection(point_inds):
    orig = np.sort(sub[np.asarray(point_inds, dtype=int)])
    utils.save_pkl(orig.astype(int), out_ind)
    print(f"✅ Saved {len(orig)} / {N} particles → {out_ind}   (retrain with cell 8.3)")
output.register_callback("cryodrgn.save_selection", _save_selection)

fig = go.Figure(go.Scattergl(
    x=emb[sub, 0], y=emb[sub, 1], mode="markers",
    marker=dict(color=labels[sub], size=3, colorscale="Turbo",
                showscale=True, colorbar=dict(title="kmeans")),
))
fig.update_layout(dragmode="lasso", width=680, height=560,
                  title=f"{emb_name} — lasso to select ({len(sub)} of {N} shown)",
                  xaxis_visible=False, yaxis_visible=False, margin=dict(l=0, r=0, t=40, b=0))

div = "cryodrgn_lasso"
html = fig.to_html(include_plotlyjs="cdn", full_html=False, div_id=div)
bridge_lines = [
    "<script>",
    "(function attach(){",
    f"  var gd = document.getElementById('{div}');",
    "  if (!gd || !gd.on) { return setTimeout(attach, 200); }",
    "  gd.on('plotly_selected', function(e){",
    "    if (!e) return;",
    "    var inds = e.points.map(function(p){ return p.pointIndex; });",
    "    google.colab.kernel.invokeFunction('cryodrgn.save_selection', [inds], {});",
    "  });",
    "})();",
    "</script>",
]
display(HTML(html + "\n".join(bridge_lines)))
print("Draw a lasso on the plot above; release to save. Re-lasso to replace the selection.")

In [ ]:
#@title 8.3 · Retrain on the filtered particles { display-mode: "form" }
#@markdown Trains a fresh model on only the kept particles via `--ind`. Leave `indices_pkl`
#@markdown blank to use the `indices.pkl` from 8.1 / 8.2, or paste a path to one you made locally
#@markdown (e.g. uploaded to Drive from a `cryodrgn filter` session).
indices_pkl = ""  #@param {type:"string"}
zdim = 8  #@param [1, 2, 4, 8, 10] {type:"raw"}
num_epochs = 25  #@param {type:"integer"}
output_name = "01_cryodrgn128_filtered"  #@param {type:"string"}
uninvert_data = False  #@param {type:"boolean"}

import os
ds = os.environ["CRYODRGN_DOWNSAMPLED"]
pose = os.environ["CRYODRGN_POSE"]
ctf = os.environ["CRYODRGN_CTF"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]

ind = indices_pkl.strip() or os.path.join(os.environ["CRYODRGN_OUTDIR"], "indices.pkl")
if not os.path.exists(ind):
    raise FileNotFoundError(f"indices.pkl not found at {ind} — run 8.1 or set indices_pkl.")
outdir = os.path.join(DRIVE_DIR, output_name)

cmd = (f'cryodrgn train_vae "{ds}" --poses "{pose}" --ctf "{ctf}" '
       f'--ind "{ind}" --zdim {zdim} -n {int(num_epochs)} -o "{outdir}"')
if uninvert_data:
    cmd += " --uninvert-data"
print("$", cmd, "\n" + "=" * 70)
get_ipython().system(cmd)

os.environ["CRYODRGN_OUTDIR"] = outdir  # Step 7 (analyze) now targets the filtered model
print("=" * 70)
print(f"✅ Filtered model → {outdir}")
print("   Re-run Step 7 (cell 7.1) — it now points at this filtered model.")

In [ ]:
#@title 8.4 · Package for interactive filtering on your local machine { display-mode: "form" }
#@markdown Bundles everything `cryodrgn filter` (or the local filtering notebook) needs into one
#@markdown zip on Drive, plus a `fix_paths.py` that repairs the paths for your machine. Download
#@markdown it, filter with the lasso locally, then bring `indices.pkl` back and retrain with 8.2.
include_particle_stack = False  #@param {type:"boolean"}
#@markdown ↑ Needed only if you trained **without CTF**, or you want the notebook's "View
#@markdown particles" montage. The stack can be several GB.

import os, shutil, yaml
from cryodrgn import config as cryodrgn_config

outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ["CRYODRGN_EPOCH"]
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]

cfg = cryodrgn_config.load(os.path.join(outdir, "config.yaml"))
da = cfg["dataset_args"]

def _locate(p):
    # tolerate Colab reconnects: try the config path, then local scratch / Drive / workdir
    if isinstance(p, str) and os.path.exists(p):
        return p
    base = os.path.basename(p) if isinstance(p, str) else None
    for d in (WORK_DIR, DRIVE_DIR, outdir):
        if base and os.path.exists(os.path.join(d, base)):
            return os.path.join(d, base)
    return None

# Stage a lean copy of the workdir: filter needs z / config / run.log + analyze umap & kmeans
# labels, but NOT the network weights or the generated .mrc volumes.
stage = os.path.join(WORK_DIR, f"filter_bundle_epoch{epoch}")
if os.path.exists(stage):
    shutil.rmtree(stage)
shutil.copytree(outdir, stage,
                ignore=shutil.ignore_patterns("weights*.pkl", "*.mrc", "*.mrcs"))

# Copy pose / ctf / ind (and optionally the stack) next to config.yaml; rewrite config to
# basenames so `cryodrgn filter .` works from inside the folder.
missing = []
for key in ("poses", "ctf", "ind", "particles"):
    if key == "particles" and not include_particle_stack:
        if isinstance(da.get(key), str):
            da[key] = os.path.basename(da[key])  # neutralize dead /content path (unused w/ CTF)
        continue
    src = _locate(da.get(key))
    if src:
        shutil.copy2(src, os.path.join(stage, os.path.basename(src)))
        da[key] = os.path.basename(src)
    elif isinstance(da.get(key), str):
        missing.append(key)

with open(os.path.join(stage, "config.yaml"), "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

# fix_paths.py upgrades the basenames to absolute paths, so the CLI and the local filtering
# notebook (whose working directory differs) both resolve regardless of where they launch.
fix_lines = [
    "# fix_paths.py - run once after unzipping (from anywhere):  python fix_paths.py",
    "# Repairs config.yaml so `cryodrgn filter` and the filtering notebook find pose.pkl /",
    "# ctf.pkl (and the particle stack, if bundled) on THIS machine.",
    "import os, yaml",
    "here = os.path.dirname(os.path.abspath(__file__))",
    "cfg_path = os.path.join(here, 'config.yaml')",
    "with open(cfg_path) as f:",
    "    cfg = yaml.safe_load(f)",
    "da = cfg['dataset_args']",
    "for key in ('poses', 'ctf', 'ind', 'particles'):",
    "    v = da.get(key)",
    "    if isinstance(v, str):",
    "        local = os.path.join(here, os.path.basename(v))",
    "        if os.path.exists(local):",
    "            da[key] = local",
    "p = da.get('particles')",
    "if isinstance(p, str) and os.path.exists(os.path.join(here, os.path.basename(p))):",
    "    da['datadir'] = here",
    "with open(cfg_path, 'w') as f:",
    "    yaml.safe_dump(cfg, f, sort_keys=False)",
    "print('Updated config.yaml paths for', here)",
]
with open(os.path.join(stage, "fix_paths.py"), "w") as f:
    f.write("\n".join(fix_lines) + "\n")

zip_base = os.path.join(DRIVE_DIR, f"filter_bundle_epoch{epoch}")
shutil.make_archive(zip_base, "zip", stage)
size_mb = os.path.getsize(zip_base + ".zip") / 1e6
shutil.rmtree(stage)

print(f"✅ Bundle → {zip_base}.zip  ({size_mb:.0f} MB)")
if missing:
    print(f"⚠️  Could not find: {', '.join(missing)} — re-run the matching Step 4 cell to regenerate.")
if da.get("ctf") is None and not include_particle_stack:
    print("⚠️  This model was trained WITHOUT CTF, so filter must open the particle stack —")
    print("    re-run this cell with include_particle_stack = True.")
print("\nOn your local machine (with cryodrgn installed):")
print(f"    unzip filter_bundle_epoch{epoch}.zip -d filter_bundle")
print( "    cd filter_bundle")
print( "    python fix_paths.py        # repair paths for this machine")
print( "    cryodrgn filter .          # interactive lasso  ->  indices.pkl")
print(f"    #  ...or open analyze.{epoch}/cryoDRGN_filtering.ipynb in local Jupyter (set EPOCH, KMEANS)")
print("\nThen upload indices.pkl to your Drive project folder and retrain with cell 8.3.")

## 9 · (Optional) Generate volumes, trajectories & landscape

Decode structures from the trained decoder: a **single volume** at a chosen `z` (9.1), a
**trajectory** — a path through latent space rendered as a volume series / movie (9.2–9.3), or a
full **conformational landscape** analysis (9.4).

In [ ]:
#@title 9.1 · Decode a volume at a specific z { display-mode: "form" }
#@markdown Space-separated latent coordinate, length == `zdim` (e.g. `0.5 -1.2 0 0 0 0 0 0`).
z_value = "0 0 0 0 0 0 0 0"  #@param {type:"string"}
#@markdown Output filename (saved in your Drive project folder).
output_name = "my_volume.mrc"  #@param {type:"string"}
#@markdown Pixel size (Å/px) for the header (`0` = default 1).
apix = 0  #@param {type:"number"}

import os
outdir = os.environ["CRYODRGN_OUTDIR"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
weights = os.path.join(outdir, "weights.pkl")
config = os.path.join(outdir, "config.yaml")
out_mrc = os.path.join(DRIVE_DIR, output_name)

cmd = f'cryodrgn eval_vol "{weights}" --config "{config}" -z {z_value} -o "{out_mrc}"'
if float(apix) > 0:
    cmd += f" --Apix {apix}"

print("$", cmd, "\n")
get_ipython().system(cmd)
print(f"\n✅ Volume → {out_mrc}")

In [ ]:
#@title 9.2 · Generate a latent trajectory (z-path) { display-mode: "form" }
#@markdown Build a path through latent space that 9.3 renders into a volume series (a movie).
#@markdown • **graph** — shortest path visiting the anchors along the data manifold (best for movies)
#@markdown • **direct** — straight-line interpolation between anchors • **pc** — along a principal component
method = "graph"  #@param ["graph", "direct", "pc"]
#@markdown **Anchors** (graph/direct) — blank = visit all k-means cluster centers; or a
#@markdown comma-separated list of **particle indices** (e.g. from a `centers_ind.txt`). Unused for `pc`.
anchors = ""  #@param {type:"string"}
#@markdown Points to sample — between each pair of anchors (`direct`) or along the PC (`pc`).
n_points = 10  #@param {type:"integer"}
#@markdown Which principal component, 1-based (`pc` method only).
pc = 1  #@param {type:"integer"}

import os, glob
outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ["CRYODRGN_EPOCH"]
zfile = os.path.join(outdir, f"z.{epoch}.pkl")
adir = os.path.join(outdir, f"analyze.{epoch}")
traj_dir = os.path.join(outdir, f"trajectory.{epoch}")
os.makedirs(traj_dir, exist_ok=True)

if method == "pc":
    cmd = f'cryodrgn pc_traversal "{zfile}" --pc {int(pc)} -n {int(n_points)} -o "{traj_dir}"'
    zpath = os.path.join(traj_dir, f"pc{int(pc)}.txt")
else:
    if anchors.strip():
        anch = anchors.replace(",", " ")
    else:
        ci = glob.glob(os.path.join(adir, "kmeans*", "centers_ind.txt"))
        if not ci:
            raise FileNotFoundError("No k-means centers_ind.txt — run 7.1 (analyze) first, or set anchors.")
        anch = f'"{ci[0]}"'
    zpath = os.path.join(traj_dir, "z-path.txt")
    if method == "graph":
        outind = os.path.join(traj_dir, "z-path-indices.txt")
        cmd = f'cryodrgn graph_traversal "{zfile}" --anchors {anch} -o "{zpath}" --outind "{outind}"'
    else:
        cmd = f'cryodrgn direct_traversal "{zfile}" --anchors {anch} -n {int(n_points)} -o "{zpath}"'

print("$", cmd, "\n")
get_ipython().system(cmd)
os.environ["CRYODRGN_ZPATH"] = zpath
npts = sum(1 for _ in open(zpath)) if os.path.exists(zpath) else 0
print(f"\n✅ z-path ({npts} points) → {zpath}   (render with 9.3)")

In [ ]:
#@title 9.3 · Render the trajectory to a volume series { display-mode: "form" }
#@markdown Decodes every z on the path into an `.mrc`. Open the folder as a volume series in
#@markdown ChimeraX for a publication movie; a quick inline slice-animation preview is shown here.
#@markdown (Graph paths can be long — rendering time scales with the number of points.)
apix = 0  #@param {type:"number"}
preview_inline = True  #@param {type:"boolean"}

import os, glob
outdir = os.environ["CRYODRGN_OUTDIR"]
zpath = os.environ["CRYODRGN_ZPATH"]
weights = os.path.join(outdir, "weights.pkl")
config = os.path.join(outdir, "config.yaml")
vol_dir = os.path.join(os.path.dirname(zpath), "volumes")

cmd = f'cryodrgn eval_vol "{weights}" --config "{config}" --zfile "{zpath}" -o "{vol_dir}"'
if float(apix) > 0:
    cmd += f" --Apix {apix}"
print("$", cmd, "\n")
get_ipython().system(cmd)

vols = sorted(glob.glob(os.path.join(vol_dir, "*.mrc")))
print(f"\n✅ {len(vols)} volumes → {vol_dir}")

if preview_inline and vols:
    import matplotlib.pyplot as plt
    from matplotlib import animation
    from IPython.display import HTML, display
    from cryodrgn.mrcfile import parse_mrc
    step = max(1, len(vols) // 60)  # cap the preview at ~60 frames
    slices = []
    for v in vols[::step]:
        vol, _ = parse_mrc(v)
        slices.append(vol[vol.shape[0] // 2].copy())  # keep only the central slice
        del vol
    fig, ax = plt.subplots(figsize=(4, 4)); ax.axis("off")
    im = ax.imshow(slices[0], cmap="Greys_r")
    def _upd(i):
        im.set_data(slices[i]); ax.set_title(f"frame {i + 1}/{len(slices)}"); return [im]
    anim = animation.FuncAnimation(fig, _upd, frames=len(slices), interval=150, blit=False)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))

In [ ]:
#@title 9.4 · (Advanced) Conformational landscape analysis { display-mode: "form" }
#@markdown `analyze_landscape` maps the whole landscape: it sketches many volumes, auto-builds a
#@markdown mask, then does volume-space PCA + clustering. Heavier than `analyze` (several minutes);
#@markdown volumes are downsampled to `-d` for speed. For the full landscape, follow with
#@markdown `cryodrgn analyze_landscape_full` on the command line.
epoch = -1  #@param {type:"integer"}
sketch_size = 1000  #@param {type:"integer"}
downsample = 128  #@param {type:"integer"}
apix = 0  #@param {type:"number"}

import os
outdir = os.environ["CRYODRGN_OUTDIR"]
if int(epoch) < 0:
    epoch = int(os.environ.get("CRYODRGN_EPOCH", "-1"))
    if epoch < 0:
        raise ValueError("Run Step 7 (analyze) first, or set epoch explicitly.")

cmd = (f'cryodrgn analyze_landscape "{outdir}" {int(epoch)} '
       f'-N {int(sketch_size)} -d {int(downsample)}')
if float(apix) > 0:
    cmd += f" --Apix {apix}"
print("$", cmd, "\n" + "=" * 70)
get_ipython().system(cmd)
print("=" * 70 + f"\n✅ Landscape → {outdir}/landscape.{int(epoch)}")

## 10 · Save & download results

Almost everything is already durable on Drive: the downsampled stack, `pose.pkl` and `ctf.pkl`
(Step 4), and all model/analysis outputs (Steps 6–9 write there directly). The only local-only
artifact is the optional back-projection map from Step 5 — copy it over here, or download any
results folder to your computer.

In [ ]:
#@title 10.1 · Back up the sanity-check map to Drive { display-mode: "form" }
#@markdown Copies the optional back-projection map (Step 5, which lives on local scratch) into
#@markdown your Drive project folder. Your stack / pose / ctf / model outputs are already there.
import os, shutil
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]

bp = os.path.join(WORK_DIR, "backproject")
if os.path.isdir(bp):
    shutil.copytree(bp, os.path.join(DRIVE_DIR, "backproject"), dirs_exist_ok=True)
    print("✅ backproject/ → Drive")
else:
    print("No local back-projection map to copy (Step 5 not run this session).")

print(f"\nEverything durable is in: {DRIVE_DIR}")

In [ ]:
#@title 10.2 · (Optional) Zip an analysis folder and download it { display-mode: "form" }
#@markdown Bundles `analyze.<epoch>/` (plots + volumes) into a zip and downloads it to your computer.
import os, shutil
from google.colab import files

outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ["CRYODRGN_EPOCH"]
adir = os.path.join(outdir, f"analyze.{epoch}")
if not os.path.isdir(adir):
    raise FileNotFoundError(f"{adir} not found — run Step 7 first.")

zip_base = os.path.join(os.environ["CRYODRGN_WORK_DIR"], f"analyze.{epoch}")
print("Zipping", adir, "...")
shutil.make_archive(zip_base, "zip", adir)
print("Starting download of", zip_base + ".zip")
files.download(zip_base + ".zip")

## 11 · Tips, troubleshooting & next steps

**Colab session limits**
- Free Colab disconnects after idle time and caps total runtime. Because per-epoch checkpoints
  are written to Drive, you can always resume with **cell 6.2** (`--load`).
- For big datasets or `D=256`, use **Colab Pro/Pro+** (A100/L4, longer sessions, more RAM).

**Common issues**
- *Back-projection / volumes look like noise* → poses or CTF likely mis-parsed. Re-check the box
  size `-D` (must be the **consensus** box, not the downsampled one) and toggle `uninvert_data`.
- *`CUDA out of memory`* → downsample to a smaller box (128), lower the batch size, or use a bigger GPU.
- *`.star`/`.cs` paths broken* → set **`datadir`** (cell 3.1) to the folder holding the `.mrcs`.
- *Slow training* → `downsample` (4.1) already writes the training stack to fast local disk; keep
  `D=128` for the first pass and only move to `D=256` once results look good.
- *Out of disk on `/content`* → free space by deleting the local downsampled stack, or work at `D=128`.

**Going further** (all available as `cryodrgn ...` commands)
- *Particle filtering* — see **Step 8**: deterministic cuts in Colab, or export a bundle for the
  interactive `cryodrgn filter` lasso locally.
- *Trajectories & landscape* — see **Step 9** (9.2–9.4): graph/PC/direct traversals rendered to a
  volume series, plus `analyze_landscape` (extend it with `analyze_landscape_full` on the CLI).
- `cryodrgn abinit_homo` / `abinit_het` — *ab-initio* reconstruction (no consensus poses needed).
- **cryoDRGN-ET** — heterogeneous subtomogram averaging for cryo-ET.

📖 Full walkthroughs: <https://ez-lab.gitbook.io/cryodrgn/> · Questions/bugs → [GitHub issues](https://github.com/ml-struct-bio/cryodrgn/issues).

---
*Volumes are best inspected in [ChimeraX](https://www.cgl.ucsf.edu/chimerax) — download the `.mrc`
files from your Drive project folder and open them there for publication-quality figures.*